Etape 1

In [2]:
with open('sample_text.txt', 'r', encoding='utf-8') as f:
    texte = f.read()

    print("Nombre de caractères :", len(texte))

    print(texte[:200])

Nombre de caractères : 2472
From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou contracted to thine own 


Etape 2

In [5]:
caracteres_uniques = sorted(set(texte))
char_to_id = {c: i for i, c in enumerate(caracteres_uniques)}
id_to_char = {i: c for c, i in char_to_id.items()}

print("Taille du vocabulaire :", len(char_to_id))

phrase = "From fairest creatures we desire increase," 
tokens = [char_to_id.get(c, 0) for c in phrase]
print("Phrase :", phrase)
print("Tokens :", tokens)

texte_decode = ''.join([id_to_char.get(t, '?') for t in tokens])
print("Texte décodé :", texte_decode)

Taille du vocabulaire : 49
Phrase : From fairest creatures we desire increase,
Tokens : [13, 40, 38, 36, 1, 30, 25, 33, 40, 29, 41, 42, 1, 27, 40, 29, 25, 42, 43, 40, 29, 41, 1, 45, 29, 1, 28, 29, 41, 33, 40, 29, 1, 33, 37, 27, 40, 29, 25, 41, 29, 3]
Texte décodé : From fairest creatures we desire increase,


Etape 3

In [ ]:
import torch
import torch.nn as nn

caracteres_uniques = sorted(set(texte))
char_to_id = {c: i for i, c in enumerate(caracteres_uniques)}
vocab_size = len(char_to_id)

data = torch.tensor([char_to_id[c] for c in texte], dtype=torch.long)

class SimpleLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x : (batch, seq_len) -> logits : (batch, seq_len, vocab_size)
        h = self.embed(x)
        h = torch.relu(self.fc(h))
        return self.out(h)

model = SimpleLLM(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

seq_len = 32  # Longueur de la fenêtre de contexte
for epoch in range(70):  # Ajustez le nombre d'epochs
    # Créez des batches (input = tokens 0..n-1, target = tokens 1..n)
    idx = torch.randint(0, len(data) - seq_len - 1, (64,))  # 64 exemples par batch
    x = torch.stack([data[i:i+seq_len] for i in idx])
    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])
    
    logits = model(x)
    loss = nn.functional.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

torch.save({'model': model.state_dict(), 'char_to_id': char_to_id}, 'model_simple.pt')

Epoch 0, Loss: 3.8958
Epoch 10, Loss: 3.4027
Epoch 20, Loss: 3.0213
Epoch 30, Loss: 2.7524
Epoch 40, Loss: 2.6083
Epoch 50, Loss: 2.4594
Epoch 60, Loss: 2.4341


Etape 4

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

checkpoint = torch.load('model_simple.pt', map_location='cpu')
char_to_id = checkpoint['char_to_id']
id_to_char = {i: c for c, i in char_to_id.items()}
vocab_size = len(char_to_id)

class SimpleLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        h = self.embed(x)
        h = torch.relu(self.fc(h))
        return self.out(h)

model = SimpleLLM(vocab_size)
model.load_state_dict(checkpoint['model'])

def generate(model, prompt, max_tokens=100, temperature=0.8):
    """
    Génère du texte à partir d'un prompt.
    - prompt : chaîne de caractères de départ
    - max_tokens : nombre de tokens à générer
    - temperature : contrôle la diversité (0 = déterministe, >1 = plus aléatoire)
    """
    model.eval()
    tokens = [char_to_id.get(c, 0) for c in prompt]
    
    with torch.no_grad():
        for _ in range(max_tokens):
            x = torch.tensor([tokens[-32:]], dtype=torch.long)  # Derniers 32 tokens
            logits = model(x)[0, -1, :]  # Logits du dernier token
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            tokens.append(next_token)
    
    return ''.join([id_to_char.get(t, '?') for t in tokens])

print(generate(model, "From ", max_tokens=100, temperature=0.8))

From re llutyeee nrand heris geare s te w su ct owoud fO w the,
Andde tree w and  t,
Buxin hef t thy foun
